# Summary using different GPT models

In this section, we will summarize the same Whisper transcript using the APIs of different well-known GPT models:
- OpenAI: GPT-3.5 Turbo 16k (openai/gpt-3.5-turbo-0125)
- OpenAI: GPT-4o (openai/gpt-4o-2024-08-06)
- OpenAI: GPT-4-mini (openai/gpt-4o-mini)
- Anthropic: Claude 3 Opus (anthropic/claude-3-opus)
- Anthropic: Claude 3 Sonnet (anthropic/claude-3-sonnet)
- Anthropic: Claude 3 Haiku (anthropic/claude-3-haiku)
- Anthropic: Claude 3.5 Sonnet (anthropic/claude-3.5-sonnet)
- Google: Gemini Pro 1.5 (google/gemini-pro-1.5)
- Google: Gemini Flash 1.5 (google/gemini-flash-1.5)
- Google: Gemma 2 27B (google/gemma-2-27b-it)
- Google: Gemma 2 9B (google/gemma-2-9b-it)
- Mistral: Mistral Large 2 (mistralai/mistral-large)
- Mistral: Mistral Medium (mistralai/mistral-medium)
- Mistral: Mistral Small (mistralai/mistral-small)
- Mistral: Mistral Tiny (mistralai/mistral-tiny)
- Mistral: Mixtral 8x22B Instruct (mistralai/mixtral-8x22b-instruct)
- Mistral: Mistral NeMo 12B (mistralai/mistral-nemo)
- Qwen: Qwen2.5 72B Instruct (qwen/qwen-2.5-72b-instruct)
- Qwen: Qwen 2 72B Instruct (qwen/qwen-2-72b-instruct)
- Qwen: Qwen 2 7B Instruct (qwen/qwen-2-7b-instruct)
- Llama 3.1 405B Instruct (llama/llama-3.1-405b-instruct)
- Llama 3.1 70B Instruct (llama/llama-3.1-8b-instruct)
- Llama 3.1 8B Instruct (llama/llama-3.1-8b-instruct)
- Hermes 3 405B Instruct (nousresearch/hermes-3-llama-3.1-405b)
- Phi-3 Medium Instruct 14B (microsoft/phi-3-medium-128k-instruct)
- Phi-3 Mini Instruct 3.8B (microsoft/phi-3-mini-128k-instruct)
- Databricks: DBRX 132B Instruct (databricks/dbrx-instruct)

We will not use the map-reduce and refine approaches (see 05_llama_on_groq.ipynb). Instead, we want to check one request and one clear response.

## Set up the environment

It is quite chalenging to interact with so many models (subscriptions, additional payments for tokens, etc). So we will use one service for this purpose: [VseGPT](https://vsegpt.ru/), which provides interactions with different models.

Install the OpenAI library:

In [1]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 968.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.0/297.0 kB 5.6 MB/s eta 0:00:00a 0:00:01


Import the libraries:

In [1]:
# Import the standard libraries
import json
import os

# Import the third party libraries
from dotenv import load_dotenv
from openai import OpenAI

Set the environment:

In [2]:
load_dotenv()

True

Test the API:

In [3]:
# Get the api and url from .env
client = OpenAI(
    api_key=os.environ.get("GPT_API_KEY"),
    base_url=os.environ.get("GPT_BASE_URL")
)
# Chat completion
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models in one sentence.",
        }
    ],
    model="openai/gpt-3.5-turbo-0125",
)

print(chat_completion.choices[0].message.content)

Fast language models are important for real-time applications such as online chatbots, voice assistants, and language translation services to provide timely and accurate responses to user queries.


## Summarize the whisper's transcript

Import the cleaned whishper transcript:

In [4]:
with open("./data/cleaned_whisper_transcript_max_context_64.txt", "r") as file:
    transcript = file.read()
transcript[0:1000]

"Welcome to IBM Think 2023. AI-generated art. AI-generated songs. AI, what is that? It sure is a lot of fun. But when foundation models are applied to big business, well, you need to think bigger. Because AI in business needs to be held to a higher standard. Built to be trusted, secured, and adaptable. This isn't simple automation that is only trained to do one thing. This is AI that is built and focused to work across your organization. This isn't committing to a single system. This is hybrid-ready AI that can scale across your systems. This isn't wondering where an answer came from. This is AI that can show its work. When you build AI into the core of your business, you can go so much further. This is more than AI. This is AI for business. Let's create. Please welcome Senior Vice President and Director of Research, IBM, Dr. Dario Gil. Hello. Welcome. Welcome. The last session of Think. And I understand some of you even had a drink. How special. So I hope you've enjoyed the last two d

Create the instruction:

In [13]:
instruction = (
    "Please divide the following text into several logical parts. "
    "The division should reflect a logical separation based on "
    "topics, themes, or any natural breakpoints in the text. "
    "Label each part with a brief heading that describes the focus or "
    "content of that section. Summarize every part. The summary every "
    "part has to be NOT less than 200 tokens.\n\n"
    "Please provide the output in the following format:\n\n"
    "Part 1: [Descriptive Heading]\n"
    "[The summary for the first section]\n\n"
    "Part 2: [Descriptive Heading]\n"
    "[The summary for the second section]\n"
)


### OpenAI: GPT-3.5 Turbo 16k (openai/gpt-3.5-turbo-0125)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [15]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "openai/gpt-3.5-turbo-0125"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to AI in Business
The text introduces the concept of AI in business, emphasizing the need for AI to be trusted, secured, and adaptable when applied to big business. It highlights the importance of foundation models and generative AI in transforming various industries and impacting every aspect of our lives. The text also stresses the significance of being an AI value creator rather than just an AI user, emphasizing the control and ownership of data and models to create value for businesses.

Part 2: Overview of WatsonX Platform
This section delves into the WatsonX platform, which is an integrated data and AI platform consisting of watsonx.data, watsonx.ai, and watsonx.governance. It explains how these components work together seamlessly to enable the creation, training, validation, tuning, and deployment of foundation models. The platform is built on Red Hat OpenShift, allowing for hybrid cloud architectures and scalable AI workloads across differe

### OpenAI: GPT-4o (openai/gpt-4o-2024-08-06)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [16]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "openai/gpt-4o-2024-08-06"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to AI in Business
The opening section of the text sets the stage for IBM Think 2023, emphasizing the transformative potential of AI in business. It highlights the shift from AI as a tool for creating art and music to a powerful force in the corporate world. The text underscores the need for AI to be trustworthy, secure, and adaptable, moving beyond simple automation to a comprehensive system that can integrate across an organization. This section introduces the concept of hybrid-ready AI, which can scale across various systems and provide transparent answers, thus building trust within business operations. The introduction concludes by welcoming Dr. Dario Gil, Senior Vice President and Director of Research at IBM, who will delve deeper into the implications and opportunities of AI in business.

Part 2: The Impact and Opportunities of AI
Dr. Dario Gil takes the stage, reflecting on the rapid advancements in AI technology and its widespread implicati

### OpenAI: GPT-4-mini (openai/gpt-4o-mini)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [17]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "openai/gpt-4o-mini"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: [Introduction to AI in Business]
The opening section of the text introduces the theme of AI's transformative potential in business, emphasizing the need for a higher standard of AI applications beyond simple automation. It highlights the importance of trust, security, and adaptability in AI systems, particularly when integrated into the core of an organization. The speaker, Dr. Dario Gil, sets the stage for the discussion by expressing excitement about the rapid advancements in AI technology and its implications across various industries. He encourages the audience to move beyond being mere users of AI to becoming value creators, emphasizing the importance of having control over AI models and data. This section establishes the foundational understanding that AI is not just a tool but a strategic asset that can drive significant value when properly harnessed.

Part 2: [Watson X: The Integrated AI Platform]
In this section, Dr. Gil introduces Watson X, IBM's inte

### Anthropic: Claude 3 Opus (anthropic/claude-3-opus)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [19]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "anthropic/claude-3-opus"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to AI and IBM Think 2023
The summary introduces the topic of AI and its growing importance in today's world. It highlights the excitement surrounding AI technology and its potential to impact every industry and aspect of our lives. The speaker, Dr. Dario Gil, emphasizes the opportunities to harness foundation models and generative AI with proper governance. He advises the audience to be AI value creators rather than just users, as value creators have control over their models and data. The introduction sets the stage for the rest of the presentation, which will delve into how IBM's WatsonX platform enables users to become AI value creators.

Part 2: Overview of WatsonX Platform
This section provides an overview of IBM's new integrated data and AI platform, WatsonX. The platform consists of three primary components: watsonx.data, watsonx.ai, and watsonx.governance. watsonx.data is a massive, curated data repository with state-of-the-art data managem

### Anthropic: Claude 3 Sonnet (anthropic/claude-3-sonnet)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [20]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "anthropic/claude-3-sonnet"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to AI and Foundation Models
[This section introduces the concept of AI and foundation models, highlighting their potential impact across various industries. It emphasizes the importance of building AI responsibly, with trust and governance at the core. The summary highlights the excitement around AI's rapid progress and the need to harness foundation models and generative AI with proper governance for immense opportunities.]

Part 2: Becoming an AI Value Creator with Watson X
[This section discusses the concept of being an AI value creator rather than just an AI user. It introduces IBM's Watson X platform, which consists of watsonx.data, watsonx.ai, and watsonx.governance, enabling users to bring their own data, train and fine-tune models, and ensure responsible AI execution. The summary walks through the end-to-end workflow of data preparation, model training, validation, tuning, and deployment using Watson X, highlighting its hybrid cloud capabil

### Anthropic: Claude 3 Haiku (anthropic/claude-3-haiku)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [21]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "anthropic/claude-3-haiku"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: The Transformative Potential of AI
The summary for the first section highlights the immense potential of AI to transform various industries and aspects of our lives. The speaker emphasizes the rapid pace of AI advancements, noting that this level of excitement and impact is a rare occurrence, occurring perhaps once or twice every decade. The speaker underscores that AI will touch every industry, from customer care to manufacturing, energy, and aerospace, and will have a profound impact on our businesses and lives. While the pace of AI can be daunting, the speaker emphasizes the vast opportunities presented by foundation models and generative AI, provided they are developed and deployed with proper governance.

Part 2: Becoming an AI Value Creator with Watson X
The summary for the second section focuses on the speaker's introduction of Watson X, IBM's integrated data and AI platform, and how it empowers users to become AI value creators rather than just AI users

### Anthropic: Claude 3.5 Sonnet (anthropic/claude-3.5-sonnet)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [22]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "anthropic/claude-3.5-sonnet"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to AI for Business
This section introduces the concept of AI for business, emphasizing the need for a higher standard of AI that is trusted, secured, and adaptable. The summary highlights the importance of AI that can work across organizations, scale across systems, and show its work. It emphasizes that AI for business is more than simple automation and encourages the audience to think bigger about AI applications in the corporate world. The introduction sets the stage for the rest of the presentation by highlighting the transformative potential of AI in various industries and the excitement surrounding this technology.

Part 2: Becoming an AI Value Creator with Watson X
This part focuses on the concept of being an AI value creator rather than just an AI user. It introduces Watson X, IBM's integrated data and AI platform, and its three main components: watsonx.data, watsonx.ai, and watsonx.governance. The summary explains the benefits of being an A

### Google: Gemini Pro 1.5 (google/gemini-pro-1.5)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [26]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "google/gemini-pro-1.5"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

## Part 1: The Importance of AI Value Creation

**Summary:** This section sets the stage by highlighting the rapid advancements in AI and its potential impact across industries. It emphasizes the need to move beyond being mere AI users and become AI value creators. The speaker argues that relying solely on external AI models limits control and ownership, while creating your own models using your data allows for customization, transparency, and ownership of the value generated. This section serves as a call to action for businesses to embrace AI value creation and control their AI destiny. 

(Word count: 208)

## Part 2: Introducing WatsonX: An AI Platform for Value Creators

**Summary:** This section introduces WatsonX, IBM's new data and AI platform designed to empower businesses to become AI value creators. It breaks down the platform into three key components: watsonx.data (a curated data repository), watsonx.ai (a studio for training and deploying AI models), and w

### Google: Gemini Flash 1.5 (google/gemini-flash-1.5)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [25]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "google/gemini-flash-1.5"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

## Part 1: AI for Business: A New Era of Value Creation
This section introduces the concept of AI for business, emphasizing its potential to go beyond simple automation and become a core component of organizational operations. It highlights the need for AI that is trusted, secure, and adaptable, capable of scaling across various systems and providing transparency in its workings. The section sets the stage for the introduction of Watson X, IBM's new integrated data and AI platform, as a tool for businesses to become AI value creators.

## Part 2: The Power of Foundation Models and Generative AI
This section delves into the transformative potential of foundation models and generative AI, emphasizing their ability to revolutionize various industries. It highlights the importance of embracing these technologies and becoming an AI value creator rather than simply an AI user. The section emphasizes the benefits of owning and controlling AI models, including the ability to l

### Google: Gemma 2 27B (google/gemma-2-27b-it)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [27]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "google/gemma-2-27b-it"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")

Bad request: Error code: 400 - {'error': {'message': "This endpoint's maximum context length is 8192 tokens. However, you requested about 9075 tokens (7027 of text input, 2048 in the output). Please reduce the length of either one.", 'code': 400}}


AttributeError: 'NoneType' object has no attribute 'choices'

It returns an error because the maximum context length is 8192. However, the same methods used in [Llama on Groq](./05_llama_on_groq.ipynb) can be applied.Returns the error because maximum context length is 8192. 

### Google: Gemma 2 9B (google/gemma-2-9b-it)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [28]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "google/gemma-2-9b-it"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")

Bad request: Error code: 400 - {'error': {'message': "This endpoint's maximum context length is 8192 tokens. However, you requested about 9075 tokens (7027 of text input, 2048 in the output). Please reduce the length of either one.", 'code': 400}}


AttributeError: 'NoneType' object has no attribute 'choices'

It returns an error because the maximum context length is 8192. However, the same methods used in [Llama on Groq](./05_llama_on_groq.ipynb) can be applied.Returns the error because maximum context length is 8192. 

### Mistral: Mistral Large 2 (mistralai/mistral-large)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [29]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "mistralai/mistral-large"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

### Part 1: Introduction to IBM Think 2023 and AI in Business

**Summary:**
The text begins with an introduction to IBM Think 2023, highlighting the excitement and potential of AI in various forms such as AI-generated art and songs. It emphasizes that AI in business needs to be held to a higher standard, focusing on trust, security, and adaptability. The AI discussed is not simple automation but is designed to work across an organization and scale across different systems. The AI should be transparent and able to show its work, making it more reliable for business applications. The introduction sets the stage for the importance of AI in transforming industries and the need for proper governance to harness its full potential.

### Part 2: The Importance of AI in Business and the Role of Foundation Models

**Summary:**
This section delves into the transformative power of AI in business, emphasizing its impact across various industries from customer care to manufacturing 

### Mistral: Mistral Medium (mistralai/mistral-medium)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [30]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "mistralai/mistral-medium"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to IBM Think 2023 and AI in Business
[The introduction to IBM Think 2023 conference highlights the fun and excitement of AI-generated art and music while emphasizing the importance of a higher standard for AI in business. The speaker explains that AI in business must be built to be trusted, secured, and adaptable, working across the organization and scaling across systems. AI in business should be able to show its work and not rely on a single system.]

Part 2: The Significance of AI and its Impact on Various Industries
[The speaker expresses their fascination with AI and its potential to impact every industry, from customer care to medicine, manufacturing, and energy. They acknowledge that while the pace of AI technology can be daunting and scary, the opportunities to harness foundation models and generative AI with proper governance are immense. The speaker encourages the audience to not just be AI users, but to become AI value creators by taking

### Mistral: Mistral Small (mistralai/mistral-small)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [31]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "mistralai/mistral-small"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

### Part 1: Introduction to IBM Think 2023 and AI in Business

The text begins with a welcoming address to the IBM Think 2023 event, focusing on the exciting advancements in AI-generated art and songs. The speaker emphasizes the importance of AI in business, highlighting the need for trust, security, and adaptability. This section introduces the concept that AI in business should be more than simple automation, emphasizing the need for AI to work across organizational structures and be scalable across various systems. The tone is enthusiastic and forward-looking, setting the stage for the event's focus on AI's transformative potential.

### Part 2: The State of AI and Its Impact on Industries

The speaker, Dr. Dario Gil, expresses his excitement about the current state of AI, noting that the technology's pace and implications are now clear for all to see. He highlights AI's widespread impact across numerous industries, including customer care, data centers, logistics, 

### Mistral: Mistral Tiny (mistralai/mistral-tiny)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [32]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "mistralai/mistral-tiny"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to IBM Think 2023 and AI in Business
[Welcome to IBM Think 2023. AI is a fun and exciting technology, but its use in business requires a higher standard of trust, security, and adaptability. This isn't simple automation, but AI that can work across an organization and be scaled across systems. The goal is to build AI into the core of business to create more value and go further. Dr. Dario Gil, Senior Vice President and Director of Research at IBM, discusses the immense possibilities of AI in various industries and the importance of being an AI value creator, not just a user.]

Part 2: Becoming an AI Value Creator with Watson X
[Dr. Gil introduces Watson X, IBM's integrated data and AI platform. It consists of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. These components work together throughout the entire lifecycle of foundation models, from data preparation and model training to deployment and governance. Watson X is buil

### Mistral: Mixtral 8x22B Instruct (mistralai/mixtral-8x22b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [33]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "mistralai/mixtral-8x22b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:



Part 1: [Welcome to IBM Think 2023]

The opening section of the text welcomes attendees to IBM Think 2023, emphasizing the importance of AI in business. It highlights the need for AI to be trusted, secured, and adaptable, and to work across an organization rather than being limited to a single system. The focus is on AI that can scale across systems and provide transparency in its operations. The section introduces the concept of AI for business, which goes beyond simple automation and commits to a higher standard of technology.

Summary:
IBM Think 2023 kicks off with a focus on the transformative role of AI in business. The event underscores the necessity for AI to be reliable, secure, and versatile, capable of integrating across various aspects of an organization. Unlike basic automation tools, the AI discussed here is designed to meet stringent business standards, offering scalability, transparency, and the ability to show its work. This approach to AI is not conf

### Mistral: Mistral NeMo 12B (mistralai/mistral-nemo)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [34]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "mistralai/mistral-nemo"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

**Part 1: IBM Think 2023 Introduction & AI Overview**
IBM Think 2023 kicked off with a discussion on AI-generated art and songs, highlighting the fun and engaging aspects of AI. However, when applied to big business, AI needs to be held to a higher standard. Dr. Dario Gil, IBM's Senior Vice President and Director of Research, emphasized that AI in business should be built to be trusted, secured, and adaptable. This isn't simple automation, but AI that works across an organization and is hybrid-ready, scalable, and transparent. Building AI into the core of your business can take you further, and IBM is here to help with AI for business.

**Part 2: IBM's AI Journey & WatsonX Platform**
IBM has had an exciting year in AI, with foundation models being the next big thing. Dr. Gil recapped IBM's AI accomplishments, including the announcement of WatsonX, a comprehensive platform for creating and governing AI in real-time. WatsonX consists of three primary parts: watsonx.data,

### Qwen: Qwen2.5 72B Instruct (qwen/qwen-2.5-72b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [36]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "qwen/qwen-2.5-72b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

### Part 1: Introduction to IBM Think 2023 and AI for Business
**Summary:**
The text begins with a welcoming note to IBM Think 2023, emphasizing the fun and excitement of AI-generated art and songs. However, it quickly shifts to the serious application of AI in business, highlighting the need for AI to be trusted, secured, and adaptable. The text stresses that AI in business is not just about simple automation but about building hybrid-ready AI that can scale across systems and show its work. The introduction sets the stage for the importance of AI in business, positioning it as more than just AI but as "AI for business." The section concludes by introducing Dr. Dario Gil, the Senior Vice President and Director of Research at IBM, who will delve deeper into the topic.

### Part 2: The Impact and Excitement of AI
**Summary:**
Dr. Dario Gil takes the stage and reflects on the incredible year for AI, noting the exhilarating pace of technological advancement and its global

### Qwen: Qwen 2 72B Instruct (qwen/qwen-2-72b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [37]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "qwen/qwen-2-72b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to IBM Think 2023 and AI for Business
[The summary for the first section]
The introduction to IBM Think 2023 highlights the fun and excitement of AI-generated art and songs, but emphasizes the importance of thinking bigger when it comes to applying AI to big business. The focus is on building AI that is trusted, secured, and adaptable, and that can work across an organization, rather than just being trained to do one thing. The session encourages attendees to think beyond simple automation and embrace AI that can show its work and be hybrid-ready, scaling across systems. The message is that when AI is built into the core of a business, it can lead to significant progress.

Part 2: The Impact of AI on Industries and the Role of Value Creators
[The summary for the second section]
This section discusses the transformative impact of AI on various industries, from customer care to medicine, manufacturing, energy, automotive, aerospace, and communication

### Qwen: Qwen 2 7B Instruct (qwen/qwen-2-7b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [38]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "qwen/qwen-2-7b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction to IBM Think 2023 and AI
Summary: The text introduces the IBM Think 2023 event, highlighting the use of AI-generated art and songs. It then transitions to discussing the importance of applying AI to big business, emphasizing the need for higher standards, trust, security, and adaptability in AI systems. The text also mentions the potential impact of AI across various industries and businesses.

Part 2: The Importance of AI in Business
Summary: The text emphasizes the transformative potential of AI in business, discussing its ability to impact industries such as customer care, data centers, logistics, medicine, manufacturing, energy, automotive, aerospace, communications, and more. It also addresses the challenges and opportunities associated with AI, advocating for responsible AI governance and value creation.

Part 3: The Role of Foundation Models and Generative AI
Summary: The text delves into the significance of foundation models and generative 

### Llama 3.1 405B Instruct (llama/llama-3.1-405b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [40]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "meta-llama/llama-3.1-405b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Here is the divided text with summaries for each part:

**Part 1: Introduction to IBM Think 2023 and AI in Business**

The text introduces IBM Think 2023 and highlights the importance of AI in business. It emphasizes that AI in business needs to be held to a higher standard, built to be trusted, secured, and adaptable. The speaker, Dr. Dario Gil, Senior Vice President and Director of Research at IBM, welcomes the audience and expresses his excitement about the possibilities that AI technology will enable. He notes that AI will impact every industry and aspect of our lives.

**Part 2: The Emergence of Foundation Models and Generative AI**

Dr. Gil discusses the emergence of foundation models and generative AI, calling it a defining moment. He advises the audience not to just be AI users, but to be AI value creators. He explains that as an AI value creator, one has control over the model, data, and can train, tune, and deploy models. He introduces Watson X, a new integra

### Llama 3.1 70B Instruct (llama/llama-3.1-70b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [41]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "meta-llama/llama-3.1-70b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Here is the divided text with brief headings and summaries for each part:

**Part 1: Introduction to IBM Think 2023 and the Importance of AI in Business**

The introduction to IBM Think 2023 highlights the excitement and potential of AI in business. The speaker emphasizes that AI is not just about automation, but about building trust, security, and adaptability. The goal is to create AI that can work across organizations and show its work, rather than just committing to a single system. The speaker introduces Dr. Dario Gil, Senior Vice President and Director of Research at IBM, who will discuss the opportunities and implications of AI.

**Summary:** The introduction sets the stage for the importance of AI in business, highlighting its potential to transform industries and create new opportunities. It emphasizes the need for trust, security, and adaptability in AI systems and introduces the main speaker, Dr. Dario Gil.

**Part 2: The Emergence of Foundation Models and G

### Llama 3.1 8B Instruct (llama/llama-3.1-8b-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [42]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "meta-llama/llama-3.1-8b-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Here are the parts of the text divided into logical sections with summaries of each part:

**Part 1: Introduction to IBM Think 2023 and AI**

The text begins with an introduction to IBM Think 2023, a conference focused on AI and its applications. The speaker, Dr. Dario Gil, welcomes the audience and highlights the excitement and importance of AI in various industries. He emphasizes that AI needs to be held to a higher standard in business, requiring trust, security, and adaptability. The speaker introduces the concept of foundation models and their potential to transform businesses.

**Summary:** The introduction sets the tone for the conference, highlighting the significance of AI and its potential to impact various industries. Dr. Gil emphasizes the need for AI to be trusted, secure, and adaptable, and introduces the concept of foundation models as a key area of focus.

**Part 2: The Importance of AI in Business**

Dr. Gil continues to discuss the importance of AI in

### Hermes 3 405B Instruct (nousresearch/hermes-3-llama-3.1-405b)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [44]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "nousresearch/hermes-3-llama-3.1-405b"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: The Emergence of Foundation Models and Generative AI
In this section, Dr. Dario Gil, Senior Vice President and Director of Research at IBM, discusses the rapid pace of AI technology and its implications for businesses. He emphasizes the importance of capturing the moment and harnessing foundation models and generative AI with proper governance. Dr. Gil advises companies to be AI value creators rather than just users, bringing their own data and models to platforms like WatsonX. He stresses the need for transparency, control, and adaptability in AI systems, as well as the importance of building AI into the core of businesses to maximize its potential.

Part 2: WatsonX: An Integrated Data and AI Platform
Dr. Gil introduces WatsonX, IBM's new integrated data and AI platform consisting of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. He explains how these components work together seamlessly throughout the entire lifecycle of foundation mode

### Phi-3 Medium Instruct 14B (microsoft/phi-3-medium-128k-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [45]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "microsoft/phi-3-medium-128k-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

 Part 1: Introduction to IBM Think 2023
In this opening section, Dr. Dario Gil, Senior Vice President and Director of Research at IBM, welcomes attendees to IBM Think 2023. He discusses the excitement surrounding AI and its rapid growth as a technology altering various industries. Gil emphasizes the importance of foundation models and generative AI, explaining how they differ from simple automation. He also encourages participants to be AI value creators instead of merely AI users. The section sets the stage for IBM's vision for AI in business, which includes trust, security, adaptability, and transparency.

Part 2: Overview of Watson X
Dr. Gil introduces Watson X, IBM's new integrated data and AI platform, designed to enable businesses to become value creators with AI. Watson X consists of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. These components work together to provide a seamless experience throughout the entire lifecycle of foundation 

### Phi-3 Mini Instruct 3.8B (microsoft/phi-3-mini-128k-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [47]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "microsoft/phi-3-mini-128k-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

 Part 1: Introduction to AI in Business and the Importance of AI for Business Growth
The session opens with Dr. Dario Gil's welcoming remarks at IBM Think 2023, emphasizing the transformative change in AI technology and its global impact across industries. Dr. Gil highlights the importance of AI in business, urging attendees to embrace AI as a tool to scale and adapt across systems, transcending simple automation. He introduces AI for business as a concept that goes beyond AI usage, focusing on its integration into core operations and its potential to drive industry transformation. The speaker encourages attendees to become AI value creators rather than passive users, suggesting that individuals can leverage their own data and models to generate value. Watson X, IBM's integrated AI platform, is introduced as a tool to empower users to become value creators, with a focus on its three main components: watsonx.data for data management, watsonx.ai for model training and tu

### Databricks: DBRX 132B Instruct (databricks/dbrx-instruct)

Generates a chat completion using the LLM API, saves the result as a JSON file, and returns the chat completion object using [summary_by_llm.py](./utils/summary_by_llm.py) :

In [48]:
from utils.summary_by_llm import summary_by_llm
api_key = os.environ.get("GPT_API_KEY")
base_url = os.environ.get("GPT_BASE_URL")
model = "databricks/dbrx-instruct"
messages = [
    {
        "role": "user",
        "content": f"{instruction}\n\nThe text:\n\n{transcript}"
    }
]
temperature = 0
output_path = os.path.join("data/llm_output", model)
chat_completion=summary_by_llm(
    api_key=api_key,
    base_url=base_url,
    model=model,
    messages=messages,
    temperature=temperature,
    output_path=output_path
)
print(f"\nFinal Summary:\n\n{chat_completion.choices[0].message.content}\n\n")
print(f"\nA number of tokens (prompt, completion, total): {chat_completion.usage.prompt_tokens}, {chat_completion.usage.completion_tokens}, {chat_completion.usage.total_tokens}\n")


Final Summary:

Part 1: Introduction and Overview of AI-Generated Art and Songs
The text begins with a discussion of the excitement surrounding AI-generated art and songs, highlighting the potential of foundation models in the business world. The speaker emphasizes the importance of holding AI to a higher standard and building it with a focus on trust, security, and adaptability. They introduce IBM's new platform, WatsonX, which is designed to work across various systems and provide transparency in AI decision-making.

Part 2: WatsonX Platform and Its Components
The second part of the text delves into the details of the WatsonX platform, which consists of three primary components: watsonx.data, watsonx.ai, and watsonx.governance. The speaker explains how these components work together to manage data, train and validate models, and ensure responsible AI deployment. They also mention that WatsonX is built on Red Hat OpenShift, allowing for seamless integration and deployment across diff

## Price comparision

|  #  | Model                                | Max Context | Price<br>(1k in/out toks),<br>RUB |     Tokens     | Tot. price,<br>RUB | ~Tot. price,<br>USD | Normalized<br> price |
| :-: | :----------------------------------- | ----------: | :-------------------------: | :------------: | --------------: | :--------------: | -------------------- |
|  1  | openai/gpt-3.5-turbo-0125            |       16385 |         0.15 -> 0.3          | 28094 -> 2626  |        2.697900 |      0.027       | 0.02                 |
|  2  | openai/gpt-4o-2024-08-06             |      128000 |         0.30 -> 1.2          | 28094 -> 6021  |       15.653400 |      0.156       | 0.14                 |
|  3  | openai/gpt-4o-mini                   |      128000 |         0.02 -> 0.08         | 28094 -> 5221  |        0.979560 |      0.010       | 0.00                 |
|  4  | anthropic/claude-3-opus              |      200000 |           2 -> 10            | 28094 -> 5031  |      106.498000 |      1.065       | 1.00                 |
|  5  | anthropic/claude-3-sonnet            |      200000 |           0.4 -> 2           | 28094 -> 2393  |       16.023600 |      0.160       | 0.15                 |
|  6  | anthropic/claude-3-haiku             |      200000 |       0.0375 -> 0.1875       | 28094 -> 3560  |        1.721025 |      0.017       | 0.01                 |
|  7  | anthropic/claude-3.5-sonnet          |      200000 |           0.4 -> 2           | 28094 -> 3828  |       18.893600 |      0.189       | 0.17                 |
|  8  | google/gemini-pro-1.5                |     1000000 |           1 -> 2.2           | 28094 -> 2556  |       33.717200 |      0.337       | 0.31                 |
|  9  | google/gemini-flash-1.5              |     1000000 |        0.017 -> 0.05         | 28094 -> 2718  |        0.613498 |      0.006       | 0.00                 |
| 10  | google/gemma-2-27b-it                |        8192 |         0.10 -> 0.10         | 28094 -> Error |               - |        -         | -                    |
| 11  | google/gemma-2-9b-it                 |        8192 |         0.03 -> 0.03         | 28094 -> Error |               - |        -         | -                    |
| 12  | mistralai/mistral-large              |      128000 |         0.45 -> 1.35         | 28094 -> 4698  |       18.984600 |      0.189       | 0.17                 |
| 13  | mistralai/mistral-medium             |       32000 |         0.42 -> 1.25         | 28094 -> 4385  |       17.280730 |      0.173       | 0.16                 |
| 14  | mistralai/mistral-small              |       32000 |         0.30 -> 0.90         | 28094 -> 4517  |       12.493500 |      0.125       | 0.11                 |
| 15  | mistralai/mistral-tiny               |       32000 |         0.04 -> 0.04         | 28094 -> 2129  |        1.208920 |      0.012       | 0.01                 |
| 16  | mistralai/mixtral-8x22b-instruct     |       65536 |         0.15 -> 0.15         | 28094 -> 9201  |        5.594250 |      0.056       | 0.05                 |
| 17  | mistralai/mistral-nemo               |      128000 |         0.05 -> 0.05         | 28094 -> 2279  |        1.518650 |      0.015       | 0.01                 |
| 18  | qwen/qwen-2.5-72b-instruct           |      128000 |         0.60 -> 0.60         | 28094 -> 5026  |       19.872000 |      0.199       | 0.18                 |
| 19  | qwen/qwen-2-72b-instruct             |       32768 |         0.14 -> 0.14         | 28094 -> 4241  |        4.526900 |      0.045       | 0.04                 |
| 20  | qwen/qwen-2-7b-instruct              |       32768 |         0.03 -> 0.03         | 28094 -> 3312  |        0.942180 |      0.009       | 0.00                 |
| 21  | llama/llama-3.1-405b-instruct        |      128000 |         0.50 -> 0.50         | 28094 -> 3133  |       15.613500 |      0.156       | 0.14                 |
| 22  | llama/llama-3.1-70b-instruct         |      128000 |         0.12 -> 0.12         | 28094 -> 4176  |        3.872400 |      0.039       | 0.03                 |
| 23  | llama/llama-3.1-8b-instruct          |      128000 |        0.027 -> 0.027        | 28094 -> 6260  |        0.927558 |      0.009       | 0.00                 |
| 24  | nousresearch/hermes-3-llama-3.1-405b |      128000 |         0.70 -> 0.70         | 28094 -> 2966  |       21.742000 |      0.217       | 0.20                 |
| 25  | microsoft/phi-3-medium-128k-instruct |      128000 |         0.15 -> 0.15         | 28094 -> 4204  |        4.844700 |      0.048       | 0.04                 |
| 26  | microsoft/phi-3-mini-128k-instruct   |      128000 |        0.015 -> 0.015        | 28094 -> 3557  |        0.474765 |      0.005       | 0.00                 |
| 27  | databricks/dbrx-instruct             |       30000 |         0.15 -> 0.15         | 28094 -> 1138  |        4.384800 |      0.044       | 0.04                 |